In [1]:
# 03_ccm_prep.ipynb
# to prepare panel crop-climate data for Convergent Cross Mapping (CCM).

In [2]:
# imports
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.preprocessing import StandardScaler

In [3]:
# paths
DATA_DIR = Path("../data")
RAW_DIR = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

INPUT_FILE = PROCESSED_DIR / "clean_panel.csv"

In [4]:
# loading data

# use if data is xlsx
# df = pd.read_excel(RAW_DIR / "final_crop_dataset-3.xlsx")

df = pd.read_csv(INPUT_FILE)

In [5]:
# basic schema check
df.head()
df.shape
df.columns.tolist()

['crop_name',
 'country_code',
 'adm_id',
 'harvest_year',
 'latitude',
 'longitude',
 'awc',
 'bulk_density',
 'drainage_class',
 'region_area',
 'crop_area',
 'crop_area_percentage',
 'sos',
 'eos',
 'yield',
 'production',
 'harvest_area',
 'avg_tmin',
 'avg_tmax',
 'avg_tavg',
 'avg_rad',
 'avg_et0',
 'avg_vpd',
 'avg_cwb',
 'avg_ssm',
 'avg_rsm',
 'avg_ndvi',
 'avg_fpar']

In [6]:
# standardizing column names
df.columns = (
    df.columns.str.strip()
              .str.lower()
              .str.replace(" ", "_")
)

In [7]:
# required columns check
required = [
    "crop_name", "adm_id", "harvest_year", "yield",
    "avg_ssm", "avg_ndvi", "avg_fpar", "avg_cwb", "avg_tavg",
    "awc", "bulk_density", "drainage_class", "latitude", "longitude"
]
missing = [c for c in required if c not in df.columns]
missing

[]

In [8]:
# this is for multiple crop, comment it out if data doesn't have more than one crop

#df = df[df["cropname"].str.lower() == "wheat"].copy()

In [9]:
# type fixing
num_cols = [
    "harvest_year", "yield", "avg_ssm", "avg_ndvi", "avg_fpar",
    "avg_cwb", "avg_tavg", "awc", "bulk_density",
    "drainage_class", "latitude", "longitude"
]

for c in num_cols:
    df[c] = pd.to_numeric(df[c], errors="coerce")

In [10]:
# sorting the panel
df = df.sort_values(["adm_id", "harvest_year"]).reset_index(drop=True)

In [11]:
# quick panel coverage
coverage = (
    df.groupby("adm_id")
      .agg(
          n_years=("harvest_year", "nunique"),
          min_year=("harvest_year", "min"),
          max_year=("harvest_year", "max")
      )
      .reset_index()
      .sort_values(["n_years", "adm_id"], ascending=[False, True])
)

coverage.head()


#This matters because CCM needs enough temporal depth per unit, 
#and the file shows annual observations by adm_id over years around 2003–2017 rather than one single long global time series.


,adm_id,n_years,min_year,max_year
2,IN-01-0051,15,2003,2017
6,IN-02-0902,15,2003,2017
7,IN-02-0903,15,2003,2017
8,IN-02-0904,15,2003,2017
9,IN-02-0905,15,2003,2017


In [12]:
# save coverage table
coverage.to_csv(PROCESSED_DIR / "panel_coverage_by_unit.csv", index=False)

In [13]:
# eligible units rule
MIN_YEARS = 10

eligible_units = coverage.loc[coverage["n_years"] >= MIN_YEARS, "adm_id"]
eligible_units = sorted(eligible_units.tolist())

len(eligible_units), eligible_units[:10]

(438,
 ['IN-01-0051',
  'IN-01-0052',
  'IN-01-0053',
  'IN-02-0902',
  'IN-02-0903',
  'IN-02-0904',
  'IN-02-0905',
  'IN-02-0906',
  'IN-02-0907',
  'IN-02-0908'])

In [14]:
# filtering eligible units
MIN_YEARS = 10

eligible_units = coverage.loc[coverage["n_years"] >= MIN_YEARS, "adm_id"]
eligible_units = sorted(eligible_units.tolist())

len(eligible_units), eligible_units[:10]

(438,
 ['IN-01-0051',
  'IN-01-0052',
  'IN-01-0053',
  'IN-02-0902',
  'IN-02-0903',
  'IN-02-0904',
  'IN-02-0905',
  'IN-02-0906',
  'IN-02-0907',
  'IN-02-0908'])

In [15]:
# creating panel
coverage = (
    df.groupby("adm_id")
      .agg(
          n_years=("harvest_year", "nunique"),
          min_year=("harvest_year", "min"),
          max_year=("harvest_year", "max")
      )
      .reset_index()
)

MIN_YEARS = 10
eligible_units = coverage.loc[coverage["n_years"] >= MIN_YEARS, "adm_id"].tolist()

panel = df[df["adm_id"].isin(eligible_units)].copy()
panel = panel.sort_values(["adm_id", "harvest_year"]).reset_index(drop=True)

panel.head()

,crop_name,country_code,adm_id,harvest_year,latitude,longitude,awc,bulk_density,drainage_class,region_area,...,avg_tmax,avg_tavg,avg_rad,avg_et0,avg_vpd,avg_cwb,avg_ssm,avg_rsm,avg_ndvi,avg_fpar
0,wheat,IN,IN-01-0051,2003,15.56859,77.82124,15.396,1.594,4,17419,...,33.499558,27.157851,2.022468e+07,5.366610,36.000909,-5.287260,2.870662,273.150972,0.369895,33.128000
1,wheat,IN,IN-01-0051,2004,15.56859,77.82124,15.396,1.594,4,17419,...,33.182636,26.735961,2.047566e+07,5.346344,35.987214,-5.182110,2.815935,273.161974,0.367158,33.340625
2,wheat,IN,IN-01-0051,2005,15.56859,77.82124,15.396,1.594,4,17419,...,33.293755,26.864381,2.056602e+07,5.275374,35.228348,-5.073142,2.813116,273.477381,0.387053,35.966375
3,wheat,IN,IN-01-0051,2006,15.56859,77.82124,15.396,1.594,4,17419,...,33.124487,26.474240,2.062392e+07,5.202429,35.939877,-4.902331,4.608838,284.335013,0.418211,39.099875
4,wheat,IN,IN-01-0051,2007,15.56859,77.82124,15.396,1.594,4,17419,...,33.349539,26.914318,2.040229e+07,5.368331,35.862955,-5.290916,4.045701,279.581649,0.382842,35.830813


In [16]:
# missingness check for dynamic vars and drop row if there are missing dynamic values
dynamic_vars = ["avg_ssm", "avg_ndvi", "avg_fpar", "avg_cwb", "avg_tavg", "yield"]

missing_summary = (
    panel.groupby("adm_id")[dynamic_vars]
         .apply(lambda x: x.isna().sum())
         .reset_index()
)

missing_summary.head()
panel = panel.dropna(subset=dynamic_vars).copy()

# recomputing coverage
coverage_post = (
    panel.groupby("adm_id")
         .agg(
             n_years=("harvest_year", "nunique"),
             min_year=("harvest_year", "min"),
             max_year=("harvest_year", "max")
         )
         .reset_index()
)

eligible_units_post = coverage_post.loc[coverage_post["n_years"] >= MIN_YEARS, "adm_id"].tolist()
panel = panel[panel["adm_id"].isin(eligible_units_post)].copy()

In [17]:
# checking continuity
def has_large_gaps(years, max_gap=1):
    ys = sorted(pd.Series(years).dropna().unique())
    if len(ys) <= 1:
        return False
    gaps = np.diff(ys)
    return (gaps > max_gap).any()

gap_table = (
    panel.groupby("adm_id")["harvest_year"]
         .apply(lambda s: has_large_gaps(s, max_gap=1))
         .reset_index(name="has_large_gap")
)

gap_table["has_large_gap"].value_counts()


#This is important because annual CCM should not silently jump across missing years inside a supposedly continuous trajectory. 
#The panel is organized by yearly observations, so continuity checking is part of making valid embeddings later.


has_large_gap
False    423
True      15
Name: count, dtype: int64

In [18]:
# optional strict continuity filter
good_units = gap_table.loc[~gap_table["has_large_gap"], "adm_id"].tolist()
panel = panel[panel["adm_id"].isin(good_units)].copy()

In [19]:
# saving eligible units
pd.DataFrame({"adm_id": sorted(panel["adm_id"].unique())}).to_csv(
    PROCESSED_DIR / "eligible_units.csv", index=False
)

In [20]:
# defining the dynamic and static var sets
dynamic_vars = ["avg_ssm", "avg_ndvi", "avg_fpar", "avg_cwb", "avg_tavg", "yield"]
static_vars = ["awc", "bulk_density", "drainage_class", "latitude", "longitude"]

In [21]:
# keep identifiers separate from features
# making panel_z
meta_cols = ["crop_name", "country_code", "adm_id", "harvest_year"]
feature_cols = ["avg_ssm", "avg_ndvi", "avg_fpar", "avg_cwb", "avg_tavg", "yield"]

meta = df[meta_cols].copy()
X = df[feature_cols].copy()

for c in feature_cols:
    X[c] = pd.to_numeric(X[c], errors="coerce")

scaler = StandardScaler()
X_z = pd.DataFrame(
    scaler.fit_transform(X),
    columns=[f"{c}_z" for c in feature_cols],
    index=df.index
)

panel_z = pd.concat([meta, X_z], axis=1)
panel_z.head()

,crop_name,country_code,adm_id,harvest_year,avg_ssm_z,avg_ndvi_z,avg_fpar_z,avg_cwb_z,avg_tavg_z,yield_z
0,wheat,IN,IN-01-0047,2017,-0.113408,1.345849,1.345125,-0.561802,1.732045,-1.151795
1,wheat,IN,IN-01-0049,2010,0.503349,1.430066,1.437267,-0.303562,1.761170,-0.710507
2,wheat,IN,IN-01-0051,2003,-1.053202,-0.857273,-0.723123,-0.977890,2.116492,-1.766069
3,wheat,IN,IN-01-0051,2004,-1.101921,-0.887430,-0.702336,-0.930119,1.965684,-1.830497
4,wheat,IN,IN-01-0051,2005,-1.104431,-0.668210,-0.445628,-0.880613,2.011589,-1.511887


In [22]:
# quick check
print(panel_z.columns.tolist())
print("adm_id" in panel_z.columns)

['crop_name', 'country_code', 'adm_id', 'harvest_year', 'avg_ssm_z', 'avg_ndvi_z', 'avg_fpar_z', 'avg_cwb_z', 'avg_tavg_z', 'yield_z']
True


In [23]:
# pair-frame function
def make_pair_frame(base, x_col, y_col, pair_name):
    keep_cols = ["crop_name", "country_code", "adm_id", "harvest_year", x_col, y_col]
    missing = [c for c in keep_cols if c not in base.columns]
    if missing:
        raise KeyError(f"{pair_name}: missing columns {missing}")

    out = base[keep_cols].copy()
    out = out.rename(columns={x_col: "x", y_col: "y"})
    out["pair"] = pair_name
    return out

In [24]:
# CCM pair construction

# build CCM pairs
pair_specs = [
    ("avg_ssm_z", "avg_ndvi_z", "ssm_ndvi"),
    ("avg_ssm_z", "avg_fpar_z", "ssm_fpar"),
    ("avg_ssm_z", "yield_z", "ssm_yield"),
    ("avg_cwb_z", "avg_ssm_z", "cwb_ssm"),
    ("avg_tavg_z", "avg_cwb_z", "tavg_cwb"),
]

pair_tables = {}
for x_col, y_col, pair_name in pair_specs:
    pair_tables[pair_name] = make_pair_frame(panel_z, x_col, y_col, pair_name)

pair_tables["ssm_ndvi"].head()

,crop_name,country_code,adm_id,harvest_year,x,y,pair
0,wheat,IN,IN-01-0047,2017,-0.113408,1.345849,ssm_ndvi
1,wheat,IN,IN-01-0049,2010,0.503349,1.430066,ssm_ndvi
2,wheat,IN,IN-01-0051,2003,-1.053202,-0.857273,ssm_ndvi
3,wheat,IN,IN-01-0051,2004,-1.101921,-0.887430,ssm_ndvi
4,wheat,IN,IN-01-0051,2005,-1.104431,-0.668210,ssm_ndvi


In [25]:
# pair metadata
ccm_pair_metadata = pd.DataFrame(
    pair_specs, columns=["x_col", "y_col", "pair_name"]
)
ccm_pair_metadata

,x_col,y_col,pair_name
0,avg_ssm_z,avg_ndvi_z,ssm_ndvi
1,avg_ssm_z,avg_fpar_z,ssm_fpar
2,avg_ssm_z,yield_z,ssm_yield
3,avg_cwb_z,avg_ssm_z,cwb_ssm
4,avg_tavg_z,avg_cwb_z,tavg_cwb


In [26]:
# pair lengths
ccm_pair_lengths = pd.DataFrame(
    [
        {
            "pair_name": name,
            "n_rows": len(tbl),
            "n_units": tbl["adm_id"].nunique(),
            "min_year": tbl["harvest_year"].min(),
            "max_year": tbl["harvest_year"].max(),
        }
        for name, tbl in pair_tables.items()
    ]
).sort_values("pair_name").reset_index(drop=True)

ccm_pair_lengths

,pair_name,n_rows,n_units,min_year,max_year
0,cwb_ssm,6715,501,2003,2017
1,ssm_fpar,6715,501,2003,2017
2,ssm_ndvi,6715,501,2003,2017
3,ssm_yield,6715,501,2003,2017
4,tavg_cwb,6715,501,2003,2017


In [27]:
# metadata and diagnostics

meta = df[["crop_name", "country_code", "adm_id", "harvest_year"]].copy()

panel_coverage_by_unit = (
    df.groupby("adm_id", as_index=False)
      .agg(
          n_years=("harvest_year", "nunique"),
          min_year=("harvest_year", "min"),
          max_year=("harvest_year", "max"),
          n_rows=("harvest_year", "size")
      )
      .sort_values(["n_years", "adm_id"], ascending=[False, True])
      .reset_index(drop=True)
)

eligible_units = panel_coverage_by_unit.loc[
    panel_coverage_by_unit["n_years"] >= 10, ["adm_id"]
].copy()

meta.head(), panel_coverage_by_unit.head(), eligible_units.head()

(  crop_name country_code      adm_id  harvest_year
 0     wheat           IN  IN-01-0047          2017
 1     wheat           IN  IN-01-0049          2010
 2     wheat           IN  IN-01-0051          2003
 3     wheat           IN  IN-01-0051          2004
 4     wheat           IN  IN-01-0051          2005,
        adm_id  n_years  min_year  max_year  n_rows
 0  IN-01-0051       15      2003      2017      15
 1  IN-02-0902       15      2003      2017      15
 2  IN-02-0903       15      2003      2017      15
 3  IN-02-0904       15      2003      2017      15
 4  IN-02-0905       15      2003      2017      15,
        adm_id
 0  IN-01-0051
 1  IN-02-0902
 2  IN-02-0903
 3  IN-02-0904
 4  IN-02-0905)

In [28]:
# saving outputs
panel_coverage_by_unit.to_csv(PROCESSED_DIR / "panel_coverage_by_unit.csv", index=False)
eligible_units.to_csv(PROCESSED_DIR / "eligible_units.csv", index=False)
ccm_pair_metadata.to_csv(PROCESSED_DIR / "ccm_pair_metadata.csv", index=False)
ccm_pair_lengths.to_csv(PROCESSED_DIR / "ccm_pair_lengths.csv", index=False)

for name, tbl in pair_tables.items():
    tbl.to_csv(PROCESSED_DIR / f"ccm_pair_{name}.csv", index=False)

print(
    """
Outputs created:
- panel_coverage_by_unit.csv
- eligible_units.csv
- ccm_pair_metadata.csv
- ccm_pair_lengths.csv
- ccm_pair_ssm_ndvi.csv
- ccm_pair_ssm_fpar.csv
- ccm_pair_ssm_yield.csv
- ccm_pair_cwb_ssm.csv
- ccm_pair_tavg_cwb.csv
"""
)


Outputs created:
- panel_coverage_by_unit.csv
- eligible_units.csv
- ccm_pair_metadata.csv
- ccm_pair_lengths.csv
- ccm_pair_ssm_ndvi.csv
- ccm_pair_ssm_fpar.csv
- ccm_pair_ssm_yield.csv
- ccm_pair_cwb_ssm.csv
- ccm_pair_tavg_cwb.csv

